In [14]:
import numpy as np
from sklearn.linear_model import LinearRegression

In [15]:
def predict(X,w,b):
    return np.dot(X,w) + b

In [16]:
def compute_cost(X, y, w, b):
    m = X.shape[0]
    residuals = predict(X,w,b) - y
    return (np.sum(residuals ** 2) / (2 * m))

In [17]:
def compute_gradient(X, y, w, b):
    m = X.shape[0]
    residuals = predict(X,w,b) - y
    Xt = np.transpose(X)
    dj_dw = np.dot(Xt,residuals) / m
    dj_db = np.sum(residuals) / m
    return dj_dw,dj_db

In [18]:
def gradient_descent(X, y, w, b, alpha, iterations):
    for i in range(iterations):
        dj_dw, dj_db = compute_gradient(X,y,w,b)
        w -= (alpha * dj_dw)
        b -= (alpha * dj_db)
    return w, b

In [19]:
np.random.seed(42)
X = np.random.randn(100, 2)

y = 2 * X[:, 0] + 3 * X[:, 1] + 1
w = np.zeros(2)
b = 0.0
w, b = gradient_descent(X,y,w,b,alpha=0.01,iterations=1000)

print(w)
print(b)

[1.99829856 3.00005581]
0.9993630361821768


In [20]:
model = LinearRegression()
model.fit(X, y)

print("sklearn w:", model.coef_)
print("sklearn b:", model.intercept_)

sklearn w: [2. 3.]
sklearn b: 0.9999999999999998


In [21]:
print("my w:", w)
print("sklearn w:", model.coef_)

print("my b:", b)
print("sklearn b:", model.intercept_)

my w: [1.99829856 3.00005581]
sklearn w: [2. 3.]
my b: 0.9993630361821768
sklearn b: 0.9999999999999998


In [22]:
import pandas as pd

df = pd.read_csv("../../data/duolingo_flagship_v5.csv")
split_df = pd.read_csv("../../data/split_users.csv")

cv_users = set(split_df.loc[split_df["split"] == "cv", "user_id"])
test_users = set(split_df.loc[split_df["split"] == "test", "user_id"])

df_cv = df[df["user_id"].isin(cv_users)].copy()
df_test = df[df["user_id"].isin(test_users)].copy()

print("Full data:", df.shape)
print("CV pool:", df_cv.shape)
print("Hold-out:", df_test.shape)
print("User overlap:", len(set(df_cv["user_id"]) & set(df_test["user_id"])))

Full data: (16382, 18)
CV pool: (14438, 18)
Hold-out: (1944, 18)
User overlap: 0


In [23]:
features = [
    "lag_days",
    "history_seen",
    "history_correct",
    "history_accuracy",
    "lag_days_log"
]

X = df_cv[features].to_numpy(dtype=float)
y = df_cv["p_recall"].to_numpy(dtype=float)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (14438, 5)
y shape: (14438,)


In [24]:
df_cv[features].describe().T[["mean", "std", "min", "max"]]

,mean,std,min,max
lag_days,8.922264,28.830331,0.0,333.920000
history_seen,16.421665,52.265991,1.0,2202.000000
history_correct,14.879692,49.400644,1.0,2136.000000
history_accuracy,0.900042,0.136223,0.2,1.000000
lag_days_log,1.066254,1.296015,0.0,5.813892


In [25]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(X_scaled.mean(axis=0))
print(X_scaled.std(axis=0))

[-2.16538858e-17 -2.85437586e-17  9.84267538e-18 -7.28850112e-16
  2.55909560e-17]
[1. 1. 1. 1. 1.]


In [26]:
w = np.zeros(X_scaled.shape[1])
b = 0.0

print(w)
print(w.shape)
print(b)

[0. 0. 0. 0. 0.]
(5,)
0.0


In [27]:
initial_cost = compute_cost(X_scaled, y, w, b)
print("Initial cost:", initial_cost)

Initial cost: 0.4377981656656927


In [28]:
w_gd = np.zeros(X_scaled.shape[1])
b_gd = 0.0

w_gd, b_gd = gradient_descent(X_scaled,y,w_gd,b_gd,alpha=0.01,iterations=1000)

final_cost = compute_cost(X_scaled, y, w_gd, b_gd)

print("Final cost:", final_cost)
print("w:", w_gd)
print("b:", b_gd)

Final cost: 0.03742710090189901
w: [ 0.00390129 -0.0035595   0.00201526  0.03045561 -0.01450211]
b: 0.8942058930266044


In [29]:
model = LinearRegression()
model.fit(X_scaled, y)

print("Sklearn w:", model.coef_)
print("Sklearn b:", model.intercept_)

Sklearn w: [ 0.00448268 -0.2456991   0.24420589  0.02572749 -0.01560702]
Sklearn b: 0.8942444986771021


In [30]:
sklearn_cost = compute_cost(X_scaled,y,model.coef_,model.intercept_)

print("GD cost:", final_cost)
print("Sklearn cost:", sklearn_cost)

GD cost: 0.03742710090189901
Sklearn cost: 0.037366558141661364


In [31]:
dj_dw, dj_db = compute_gradient(X_scaled, y, w_gd, b_gd)

print("Gradient norm:", np.linalg.norm(dj_dw))
print("db:", dj_db)

Gradient norm: 0.00040473330849859473
db: -3.860565049775556e-05


In [32]:
sk_dw, sk_db = compute_gradient(X_scaled,y,model.coef_,model.intercept_)

print("Sklearn gradient norm:", np.linalg.norm(sk_dw))
print("Sklearn db:", sk_db)

Sklearn gradient norm: 2.872364550292664e-16
Sklearn db: -7.87414030486328e-18


In [33]:
w_gd_10k = np.zeros(X_scaled.shape[1])
b_gd_10k = 0.0

w_gd_10k, b_gd_10k = gradient_descent(X_scaled,y,w_gd_10k,b_gd_10k,alpha=0.01,iterations=10000)

cost_10k = compute_cost(X_scaled, y, w_gd_10k, b_gd_10k)
dw_10k, db_10k = compute_gradient(X_scaled, y, w_gd_10k, b_gd_10k)

print("Cost:", cost_10k)
print("w:", w_gd_10k)
print("b:", b_gd_10k)
print("Gradient norm:", np.linalg.norm(dw_10k))
print("db:", db_10k)

Cost: 0.0374167914545275
w: [ 0.00437322 -0.02502566  0.02346924  0.03003467 -0.0150218 ]
b: 0.8942444986770967
Gradient norm: 0.00032184918746738377
db: -5.458009565692888e-15


In [34]:
corr = pd.DataFrame(X_scaled, columns=features).corr()
corr

,lag_days,history_seen,history_correct,history_accuracy,lag_days_log
lag_days,1.000000,-0.042027,-0.040086,0.029535,0.696774
history_seen,-0.042027,1.000000,0.998775,0.013976,-0.075932
history_correct,-0.040086,0.998775,1.000000,0.033511,-0.073155
history_accuracy,0.029535,0.013976,0.033511,1.000000,0.023192
lag_days_log,0.696774,-0.075932,-0.073155,0.023192,1.000000


In [35]:
m = X_scaled.shape[0]

H = (X_scaled.T @ X_scaled) / m

eigenvalues = np.linalg.eigvalsh(H)

print("Eigenvalues:", eigenvalues)
print("Condition number:", eigenvalues.max() / eigenvalues.min())

Eigenvalues: [1.03105781e-03 3.02541588e-01 9.96610530e-01 1.66090797e+00
 2.03890885e+00]
Condition number: 1977.4922648530874


In [36]:
pred_gd = predict(X_scaled, w_gd_10k, b_gd_10k)
pred_sk = model.predict(X_scaled)

print("Max prediction difference:", np.max(np.abs(pred_gd - pred_sk)))
print("GD RMSE: ", np.sqrt(np.mean((pred_gd - y) ** 2)))
print("Sklearn RMSE: ",np.sqrt(np.mean((pred_sk - y) ** 2)))

Max prediction difference: 0.2482540794746877
GD RMSE:  0.2735572753721878
Sklearn RMSE:  0.2733735837335472


In [37]:
w_gd_fast = np.zeros(X_scaled.shape[1])
b_gd_fast = 0.0

w_gd_fast, b_gd_fast = gradient_descent(X_scaled,y,w_gd_fast,b_gd_fast,alpha=0.1,iterations=10000)

cost_fast = compute_cost(X_scaled, y, w_gd_fast, b_gd_fast)
dw_fast, db_fast = compute_gradient(X_scaled, y, w_gd_fast, b_gd_fast)

print("Cost:", cost_fast)
print("w:", w_gd_fast)
print("b:", b_gd_fast)
print("Gradient norm:", np.linalg.norm(dw_fast))

Cost: 0.03737440936290939
w: [ 0.0044394  -0.15845767  0.15693947  0.0274303  -0.01537566]
b: 0.8942444986771017
Gradient norm: 0.00012724042608029418


In [38]:
pred_fast = predict(X_scaled, w_gd_fast, b_gd_fast)

print("Max prediction difference:",np.max(np.abs(pred_fast - pred_sk)))
print("GD fast RMSE:",np.sqrt(np.mean((pred_fast - y) ** 2)))
print("Sklearn RMSE:",np.sqrt(np.mean((pred_sk - y) ** 2)))

Max prediction difference: 0.09814520613550726
GD fast RMSE: 0.2734023019760784
Sklearn RMSE: 0.2733735837335472


## Conclusion

I generalized my from-scratch gradient descent implementation from a single feature to multiple features and validated it first on synthetic data, where it recovered the known coefficients and matched `sklearn.linear_model.LinearRegression`.

On the flagship dataset, the features had very different scales, so I standardized them before optimization. Gradient descent reduced the cost from approximately `0.438` to `0.0374`.

The first runs converged slowly even after scaling. Investigation showed that `history_seen` and `history_correct` were almost perfectly correlated (`r ≈ 0.9988`). The Hessian had a condition number of approximately `1977`, indicating an ill-conditioned optimization problem with very different curvature across parameter-space directions.

Increasing the learning rate from `0.01` to `0.1` substantially improved convergence. The final from-scratch model achieved an in-sample RMSE of approximately `0.27340`, compared with `0.27337` for sklearn.

These RMSE values are not generalization estimates because the model was evaluated on the same CV pool used for fitting. Proper group-aware validation will be performed later.